In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import traceback

import numpy as np
import pandas as pd
import scipy
from scipy.stats import t as student_t
import sklearn
from sklearn.model_selection import GroupKFold
import statsmodels
import statsmodels.api as sm
import xgboost
from xgboost import XGBRegressor
import time

# ======================== User configuration ========================
INPUT_DIR = Path("../outputs/final_panel")
OUT_BASE = Path("../outputs/dml")
ALL_SCHEMES = ["local_p90", "local_p95", "city_p90", "city_p95", "fixed_tx35_tn25"]
SCHEMES = ALL_SCHEMES.copy()
N_REPEATS = 10
RANDOM_STATE = 42
CV_FOLDS = 3
MIN_N = 200
N_ESTIMATORS = 250
N_JOBS = 4
YEARS = list(range(2016, 2024))
KEY = ["city", "cell_id", "year"]
HEAT_TYPES = ["day", "night", "compound"]
TREATMENTS = {"bld_coverage_z": "Building density", "bld_mean_h_m_z": "Mean building height"}
BASE_CONTROLS = ["lat", "dem_mean_m", "dem_mean_slope", "coast_dist_km",
                 "rh_mean_pct", "tp_annual_mm", "cdd_22C_city", "water_coverage_z"]
EXTRA_CONTROLS = ["ndvi_mean_z", "isa_mean_prob_z", "dist_to_cbd_km"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def write_json(path, value):
    def conv(x):
        if isinstance(x, Path): return str(x)
        if isinstance(x, np.generic): return x.item()
        raise TypeError(type(x).__name__)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=conv), encoding="utf-8")


def boolean(s):
    if pd.api.types.is_bool_dtype(s):
        require(s.notna().all(), f"Boolean flag has missing values: {s.name}")
        return s.astype(bool)
    x = s.astype(str).str.strip().str.lower().map({"true": True, "false": False, "1": True, "0": False})
    require(x.notna().all(), f"Cannot parse boolean flag: {s.name}")
    return x.astype(bool)


def finite(d, cols):
    missing = sorted(set(cols) - set(d.columns))
    require(not missing, f"Required variables missing, not dropped automatically: {missing}")
    return pd.Series(np.isfinite(d[cols].to_numpy(float)).all(axis=1), index=d.index)


def sample_stats(d):
    return dict(n=len(d), n_cities=d.city.nunique(),
                n_cells=d[["city", "cell_id"]].drop_duplicates().shape[0],
                n_city_years=d[["city", "year"]].drop_duplicates().shape[0])


def load_panel(scheme):
    path = INPUT_DIR / f"panel_final_{scheme}_2016_2023.csv"
    require(path.is_file(), f"Input does not exist: {path}")
    d = pd.read_csv(path)
    require(not d.duplicated(KEY).any(), f"Duplicate city/cell_id/year: {path}")
    require(d[KEY].notna().all().all(), "Sample keys have missing values")
    require(set(d.year.unique()) == set(YEARS), "Study years must be 2016–2023")
    require(len(d) == 5216 and d.city.nunique() == 35, "Input must be the unfiltered main panel with 35 cities and 5216 rows")
    require(d[["city", "cell_id"]].drop_duplicates().shape[0] == 652, "Number of study grid cells is not 652")
    require(d.groupby(["city", "cell_id"]).year.nunique().eq(8).all(), "Study grid cell is missing years")
    require(d.scheme.eq(scheme).all(), f"scheme label does not match the file name: {scheme}")
    for c in ["sample_dml_controls", "sample_dml_context_controls",
              "heatwave_annual_complete", "eligible_density_model", "eligible_height_model",
              "any_main_input_imputed"] + [f"sample_dml_{h}_positive_log" for h in HEAT_TYPES]:
        require(c in d, f"Missing sample flag {c}")
        d[c] = boolean(d[c])
    d = d.sort_values(KEY).reset_index(drop=True)
    d["_cell_key"] = d.city.astype(str) + "|" + d.cell_id.astype(str)
    base = d.sample_dml_controls
    require((~base | (d.heatwave_annual_complete & d.eligible_density_model & d.eligible_height_model)).all(),
            "DML flag includes rows that fail the heatwave/building quality conditions")
    require((~base | finite(d, list(TREATMENTS) + BASE_CONTROLS)).all(), "Invalid controls within the DML flag")
    context = d.sample_dml_context_controls
    require(context.equals(base & finite(d, EXTRA_CONTROLS)), "context sample flag is inconsistent with variable completeness")
    p = pd.to_numeric(d.pop_study, errors="raise")
    require((~base | (np.isfinite(p) & p.gt(0))).all(), "Population non-positive or missing within the DML flag")
    require(np.allclose(p, d.pop_total, equal_nan=True), "pop_total does not match pop_study, the population within the boundary")
    d["log_pop"] = np.log1p(p)
    for h in HEAT_TYPES:
        hwd = pd.to_numeric(d[f"HW_days_{h}"], errors="raise")
        exp = pd.to_numeric(d[f"exposure_people_days_{h}"], errors="raise")
        require((~base | (np.isfinite(hwd) & hwd.ge(0))).all(), f"Invalid HWD within the DML flag: {h}")
        require((hwd.dropna() >= 0).all(), f"Negative HWD values: {h}")
        require(np.allclose(exp, p * hwd, rtol=1e-10, atol=1e-7, equal_nan=True), f"E != P*HWD: {h}")
        require(np.allclose(d[f"log_exp_{h}"], np.log1p(exp), rtol=1e-10, atol=1e-10, equal_nan=True),
                f"Saved log_exp is not log1p(E): {h}")
        d[f"log_hwd_{h}"] = np.log1p(hwd)
        pos = base & p.gt(0) & hwd.gt(0) & exp.gt(0) & np.isfinite(exp)
        require(pos.equals(d[f"sample_dml_{h}_positive_log"]), f"Wrong positive-sample flag: {h}")
    return d, path


def main_outcomes():
    return ([f"log_exp_{h}" for h in HEAT_TYPES] + ["log_pop"] +
            [f"log_hwd_{h}" for h in HEAT_TYPES])


def select_sample(d, variant="base", positive_heat=None):
    flag = "sample_dml_context_controls" if variant == "base_plus_context" else "sample_dml_controls"
    mask = d[flag].copy()
    if positive_heat is not None:
        mask &= d[f"sample_dml_{positive_heat}_positive_log"]
    s = d.loc[mask].copy().reset_index(drop=True)
    if positive_heat is not None:
        h = positive_heat
        s["ln_population"] = np.log(s.pop_study)
        s["ln_hwd"] = np.log(s[f"HW_days_{h}"])
        s["ln_exposure"] = np.log(s[f"exposure_people_days_{h}"])
        require(np.allclose(s.ln_exposure, s.ln_population + s.ln_hwd, rtol=0, atol=1e-12),
                "Check of ln(E)=ln(P)+ln(HWD) failed on the positive sample")
    return s


def build_W(d, treatment, variant):
    require(variant in ["base", "base_plus_context", "base_no_mutual_morphology"], "Unknown control-variable variant")
    cols = BASE_CONTROLS.copy()
    if variant != "base_no_mutual_morphology":
        cols += [t for t in TREATMENTS if t != treatment]
    if variant == "base_plus_context":
        cols += EXTRA_CONTROLS
    require(finite(d, cols).all(), "W has missing or infinite values after filtering; no median filling is done")
    W = d[cols].astype(float).reset_index(drop=True)
    cat = d[["city", "year"]].astype(str)
    fe = pd.get_dummies(cat, drop_first=True, dtype=float).reset_index(drop=True)
    W = pd.concat([W, fe], axis=1)
    require(np.isfinite(W.to_numpy()).all(), "W has non-finite values")
    return W, cols


def folds_for(d):
    require(d._cell_key.nunique() >= CV_FOLDS, "Not enough study grid cells for cross-fitting")
    folds = list(GroupKFold(n_splits=CV_FOLDS).split(d, groups=d._cell_key))
    assigned = np.full(len(d), -1, dtype=int)
    reports = []
    for k, (train, test) in enumerate(folds):
        require(not set(d.iloc[train]._cell_key) & set(d.iloc[test]._cell_key), "Same grid cell spans training and test folds")
        assigned[test] = k
        unseen = sorted(set(d.iloc[test].city) - set(d.iloc[train].city))
        reports.append(dict(fold=k, n_train=len(train), n_test=len(test),
                            train_cities=d.iloc[train].city.nunique(), test_cities=d.iloc[test].city.nunique(),
                            unseen_test_cities=len(unseen), unseen_city_names=";".join(unseen),
                            n_test_rows_unseen_city=int(d.iloc[test].city.isin(unseen).sum())))
    require((assigned >= 0).all(), "Some rows received no OOF prediction")
    require(pd.DataFrame({"cell": d._cell_key, "fold": assigned}).groupby("cell").fold.nunique().eq(1).all(),
            "Different years of the same grid cell span folds")
    return folds, assigned, reports


def new_learner():
    return XGBRegressor(n_estimators=N_ESTIMATORS, max_depth=3, learning_rate=0.04,
                        subsample=0.85, colsample_bytree=0.85, objective="reg:squarederror",
                        random_state=RANDOM_STATE, n_jobs=N_JOBS, tree_method="hist", verbosity=0)


def predict_oof(W, target, folds):
    y = np.asarray(target, dtype=float)
    require(np.isfinite(y).all(), "Target has missing values")
    pred = np.full(len(y), np.nan)
    for train, test in folds:
        model = new_learner()
        model.fit(W.iloc[train], y[train])
        pred[test] = model.predict(W.iloc[test])
    require(np.isfinite(pred).all(), "OOF predictions do not cover all rows")
    return pred


def oof_r2(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    total = np.sum((y-y.mean())**2)
    return float(1-np.sum((y-pred)**2)/total) if total > 1e-20 else np.nan


def inference_fields(theta, se, df, suffix):
    good = np.isfinite(se) and se > 0 and df > 0
    return {f"SE_{suffix}": float(se),
            f"CI_lo_{suffix}": float(theta-student_t.ppf(0.975, df)*se) if good else np.nan,
            f"CI_hi_{suffix}": float(theta+student_t.ppf(0.975, df)*se) if good else np.nan,
            f"pval_{suffix}": float(2*student_t.sf(abs(theta/se), df)) if good else np.nan}


def final_stage(y, t, cities):
    y, t = np.asarray(y, dtype=float), np.asarray(t, dtype=float)
    require(np.isfinite(y).all() and np.isfinite(t).all(), "Final residuals contain missing values")
    require(np.std(t) > 1e-10, "Treatment residual has almost no variation; cannot estimate")
    codes, labels = pd.factorize(cities, sort=True)
    G = len(labels)
    require(G >= 2, "City clustering needs at least two cities")
    fit = sm.OLS(y, sm.add_constant(t, has_constant="add")).fit()
    robust = fit.get_robustcov_results(cov_type="cluster", groups=codes,
                                      use_correction=True, df_correction=True, use_t=True)
    theta = float(fit.params[1])
    result = dict(theta=theta, intercept=float(fit.params[0]), n_clusters=G,
                  few_clusters_warning=G < 20, cluster_df=G-1,
                  **inference_fields(theta, fit.bse[1], fit.df_resid, "iid"),
                  **inference_fields(theta, robust.bse[1], G-1, "cluster"))
    for field in ["SE", "CI_lo", "CI_hi", "pval"]:
        result[field+"_main"] = result[field+"_cluster"]
    return result


def preflight():
    require(bool(SCHEMES) and len(set(SCHEMES)) == len(SCHEMES) and set(SCHEMES)<=set(ALL_SCHEMES), "Invalid SCHEMES setting")
    panels, manifest, counts = {}, [], []
    for scheme in SCHEMES:
        d, path = load_panel(scheme)
        panels[scheme] = d
        manifest.append(dict(scheme=scheme, path=str(path), sha256=digest(path)))
        for variant in ["base", "base_plus_context", "base_no_mutual_morphology"]:
            s = select_sample(d, variant)
            counts.append(dict(scheme=scheme, variant=variant, analysis="main_log1p", heat="all", **sample_stats(s)))
            for h in HEAT_TYPES:
                s = select_sample(d, variant, positive_heat=h)
                counts.append(dict(scheme=scheme, variant=variant, analysis="positive_ln", heat=h, **sample_stats(s)))
    return panels, manifest, pd.DataFrame(counts)


def run_analysis():
    panels, manifest, counts = preflight()
    out = OUT_BASE
    out.mkdir(parents=True, exist_ok=True)
    config = {name: globals()[name] for name in ["INPUT_DIR", "SCHEMES",
              "RANDOM_STATE", "CV_FOLDS", "MIN_N", "N_REPEATS",
              "N_ESTIMATORS", "N_JOBS", "BASE_CONTROLS", "EXTRA_CONTROLS"]}
    config["versions"] = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                              scipy=scipy.__version__, sklearn=sklearn.__version__,
                              statsmodels=statsmodels.__version__, xgboost=xgboost.__version__)
    write_json(out/"configuration.json", config)
    pd.DataFrame(manifest).to_csv(out/"input_manifest.csv", index=False)
    counts.to_csv(out/"sample_counts_preflight.csv", index=False)
    transforms = []
    for scheme, d in panels.items():
        s = select_sample(d)
        for h in HEAT_TYPES:
            gap = s[f"log_exp_{h}"] - s.log_pop - s[f"log_hwd_{h}"]
            transforms.append(dict(scheme=scheme, heat=h, n=len(s),
                                   n_zero_hwd=int(s[f"HW_days_{h}"].eq(0).sum()),
                                   fraction_zero_hwd=float(s[f"HW_days_{h}"].eq(0).mean()),
                                   log1p_identity_gap_mean=float(gap.mean()),
                                   log1p_identity_gap_min=float(gap.min()),
                                   log1p_identity_gap_max=float(gap.max())))
    pd.DataFrame(transforms).to_csv(out/"log1p_zero_HWD_diagnostic.csv", index=False)
    print("Output folder:", out, flush=True)
    write_json(out/"summary.json", dict(status="running", output=str(out)))
    results, checks, specs = [], [], []
    def run_one(s, scheme, analysis, variant, outcomes, label, heat=None):
        name = f"{scheme}__{analysis}__{variant}__{label}"
        r, c, sp = fit_block_repeated(s, scheme, analysis, variant, outcomes, label, out/"blocks"/name,
                                      positive_heat=heat)
        results.extend(r); checks.extend(c); specs.extend(sp)
        pd.DataFrame(results).to_csv(out/"dml_all_results.csv", index=False)
        pd.DataFrame(specs).to_csv(out/"model_specification.csv", index=False)
        if checks: pd.DataFrame(checks).to_csv(out/"positive_log_additivity_checks.csv", index=False)
    try:
        for scheme, d in panels.items():
            variants = ["base", "base_plus_context", "base_no_mutual_morphology"]
            for variant in variants:
                run_one(select_sample(d, variant), scheme, "main_log1p", variant,
                        main_outcomes(), "full")
            for h in HEAT_TYPES:
                run_one(select_sample(d, "base", positive_heat=h), scheme, "positive_ln", "base",
                        ["ln_exposure", "ln_population", "ln_hwd"], h, heat=h)
            for j, (zone, s) in enumerate(select_sample(d).groupby("koppen_broad", dropna=False)):
                require(pd.notna(zone), "Climate grouping has missing labels")
                run_one(s.reset_index(drop=True), scheme, "climate_log1p", "base", main_outcomes(), f"zone{j}")
                (out/"blocks"/f"{scheme}__climate_log1p__base__zone{j}"/"climate_zone.txt").write_text(str(zone), encoding="utf-8")
        result = pd.DataFrame(results)
        if not result.empty:
            result[result.analysis.eq("main_log1p")].to_csv(out/"dml_main_results.csv", index=False)
            result[result.analysis.eq("positive_ln")].to_csv(out/"dml_positive_log_results.csv", index=False)
        unchanged = all(digest(m["path"]) == m["sha256"] for m in manifest)
        require(unchanged, "Input files changed during the run, please check")
        write_json(out/"summary.json", dict(status="completed", output=str(out), schemes=SCHEMES,
                   result_rows=len(results), additivity_checks=len(checks), inputs_unchanged=unchanged,
                   n_ok=sum(r["status"]=="ok" for r in results),
                   n_skipped=sum(r["status"]!="ok" for r in results)))
        return out, result, pd.DataFrame(checks)
    except (Exception, KeyboardInterrupt):
        write_json(out/"summary.json", dict(status="failed", output=str(out), completed_result_rows=len(results),
                                            traceback=traceback.format_exc()))
        raise


# ======================== Repeated cross-fitting ========================
def folds_for_repeat(d, r):
    if r == 0:
        return folds_for(d)
    rng = np.random.default_rng(RANDOM_STATE + r)
    cells = d._cell_key.unique()
    code = dict(zip(rng.permutation(cells), range(len(cells))))
    groups = d._cell_key.map(code).to_numpy()
    folds = list(GroupKFold(n_splits=CV_FOLDS).split(d, groups=groups))
    assigned = np.full(len(d), -1, dtype=int)
    reports = []
    for k, (train, test) in enumerate(folds):
        require(not set(d.iloc[train]._cell_key) & set(d.iloc[test]._cell_key), "Same grid cell spans training and test folds")
        assigned[test] = k
        unseen = sorted(set(d.iloc[test].city) - set(d.iloc[train].city))
        reports.append(dict(fold=k, n_train=len(train), n_test=len(test),
                            train_cities=d.iloc[train].city.nunique(), test_cities=d.iloc[test].city.nunique(),
                            unseen_test_cities=len(unseen), unseen_city_names=";".join(unseen),
                            n_test_rows_unseen_city=int(d.iloc[test].city.isin(unseen).sum())))
    require((assigned >= 0).all(), "Some rows received no OOF prediction")
    return folds, assigned, reports


def aggregate(est, n, G):
    th = np.array([e["theta"] for e in est], dtype=float)
    med = float(np.median(th))
    out = dict(theta=med, theta_min_over_splits=float(th.min()), theta_max_over_splits=float(th.max()),
               theta_sd_over_splits=float(th.std(ddof=1)) if len(th) > 1 else 0.0,
               n_splits=len(est), n_clusters=G, cluster_df=G - 1, few_clusters_warning=G < 20,
               intercept=float(np.median([e["intercept"] for e in est])))
    for suffix, df in [("iid", n - 2), ("cluster", G - 1)]:
        se = np.array([e[f"SE_{suffix}"] for e in est], dtype=float)
        se_med = float(np.sqrt(np.median(se ** 2 + (th - med) ** 2)))
        out.update(inference_fields(med, se_med, df, suffix))
        out[f"share_splits_p05_{suffix}"] = float(np.mean([e[f"pval_{suffix}"] < 0.05 for e in est]))
    for f in ["SE", "CI_lo", "CI_hi", "pval"]:
        out[f + "_main"] = out[f + "_cluster"]
    return out


def fit_block_repeated(d, scheme, analysis, variant, outcomes, sample_label, folder,
                       positive_heat=None):
    folder.mkdir(parents=True, exist_ok=True)
    stats = sample_stats(d)
    common = dict(scheme=scheme, analysis=analysis, w_variant=variant, sample=sample_label,
                  heatwave_type=positive_heat or "all", **stats,
                  transform="ln_positive" if positive_heat else "log1p",
                  categorical_controls="city_and_year_dummies",
                  cv_group="city|cell_id", cv_folds=CV_FOLDS, n_used=len(d))
    require(finite(d, outcomes + list(TREATMENTS)).all(), "Outcome/treatment still have missing values after filtering")
    if len(d) < MIN_N or d.city.nunique() < 2 or d._cell_key.nunique() < CV_FOLDS:
        rows = [{**common, "treatment": t, "treatment_label": lab, "outcome": y,
                 "status": "skipped_insufficient_sample", "theta": np.nan}
                for t, lab in TREATMENTS.items() for y in outcomes]
        pd.DataFrame(rows).to_csv(folder / "results.csv", index=False)
        return rows, [], []

    splits, audits = [], []
    for r in range(N_REPEATS):
        folds, assigned, reports = folds_for_repeat(d, r)
        splits.append(folds)
        audits += [{**x, "split": r} for x in reports]
    pd.DataFrame(audits).to_csv(folder / "fold_audit_all_splits.csv", index=False)
    unseen = sum(a["n_test_rows_unseen_city"] for a in audits)
    common["n_oof_rows_unseen_city_all_splits"] = unseen
    common["unseen_city_prediction_warning"] = unseen > 0
    G = d.city.nunique()

    rows, checks, specs, per_split = [], [], [], []
    for treatment, label in TREATMENTS.items():
        print(f"  {scheme} | {analysis} | {sample_label} | {variant} | {label} | n={len(d)} | x{N_REPEATS}", flush=True)
        W, numeric = build_W(d, treatment, variant)
        T = d[treatment].to_numpy(float)
        est = {y: [] for y in outcomes}
        gap_est, r2y, r2t, rsd = [], {y: [] for y in outcomes}, [], []
        for r, folds in enumerate(splits):
            t_hat = predict_oof(W, T, folds)
            tr = T - t_hat
            r2t.append(oof_r2(T, t_hat))
            rsd.append(float(np.std(tr) / np.std(T)) if np.std(T) > 0 else np.nan)
            res = {}
            for y in outcomes:
                yy = d[y].to_numpy(float)
                yh = predict_oof(W, yy, folds)
                res[y] = yy - yh
                e = final_stage(res[y], tr, d.city)
                est[y].append(e)
                r2y[y].append(oof_r2(yy, yh))
                per_split.append(dict(treatment=treatment, outcome=y, split=r, theta=e["theta"],
                                      SE_cluster=e["SE_cluster"], pval_cluster=e["pval_cluster"],
                                      SE_iid=e["SE_iid"], pval_iid=e["pval_iid"]))
            if positive_heat:
                gap = res["ln_exposure"] - res["ln_population"] - res["ln_hwd"]
                gap_est.append(final_stage(gap, tr, d.city))
        agg = {}
        for y in outcomes:
            agg[y] = aggregate(est[y], len(d), G)
            rows.append({**common, **agg[y], "treatment": treatment, "treatment_label": label, "outcome": y,
                         "status": "ok", "R2_outcome": float(np.median(r2y[y])),
                         "R2_treatment": float(np.median(r2t)), "resid_sd_ratio_T": float(np.median(rsd)),
                         "outcome_constant": bool(np.ptp(d[y].to_numpy(float)) == 0)})
        specs.append({**common, "treatment": treatment, "numeric_controls": ";".join(numeric),
                      "outcomes": ";".join(outcomes), "n_W": W.shape[1],
                      "mutual_morphology": variant != "base_no_mutual_morphology"})
        if positive_heat:
            g = aggregate(gap_est, len(d), G)
            raw_gap = d.ln_exposure - d.ln_population - d.ln_hwd
            s_ = agg["ln_population"]["theta"] + agg["ln_hwd"]["theta"]
            checks.append({**common, "treatment": treatment, "treatment_label": label,
                           "raw_identity_max_abs_error": float(np.max(np.abs(raw_gap))),
                           "theta_exposure_independent": agg["ln_exposure"]["theta"],
                           "theta_population": agg["ln_population"]["theta"], "theta_hwd": agg["ln_hwd"]["theta"],
                           "theta_components_sum": s_,
                           "theta_exposure_minus_sum_of_medians": agg["ln_exposure"]["theta"] - s_,
                           "gap_theta_median": g["theta"], "gap_SE_cluster": g["SE_cluster"],
                           "gap_CI_lo_cluster": g["CI_lo_cluster"], "gap_CI_hi_cluster": g["CI_hi_cluster"],
                           "gap_pval_cluster": g["pval_cluster"]})
        pd.DataFrame(rows).to_csv(folder / "results.csv", index=False)
    pd.DataFrame(per_split).to_csv(folder / "per_split_estimates.csv", index=False)
    pd.DataFrame(specs).to_csv(folder / "specification.csv", index=False)
    if checks:
        pd.DataFrame(checks).to_csv(folder / "log_additivity_checks.csv", index=False)
    return rows, checks, specs


def run_repeated():
    require(N_REPEATS >= 1, "N_REPEATS must be at least 1")
    t0 = time.time()
    out, result, checks = run_analysis()
    (Path(out) / "repeated_crossfit_configuration.json").write_text(json.dumps(dict(
        N_REPEATS=N_REPEATS,
        runtime_minutes=(time.time() - t0) / 60),
        ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Repeated cross-fitting finished: {out} (took {(time.time() - t0) / 60:.1f} minutes)")
    return out, result, checks


DML_RCF_OUTPUT_DIR, DML_RCF_RESULTS, DML_RCF_CHECKS = run_repeated()